# Stage 7: Model Evaluation & Comparison

Comprehensive evaluation: confusion matrices, ROC curves, precision-recall curves, and model comparison table.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve,
    f1_score, average_precision_score
)
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Evaluation notebook ready.')

In [ ]:
# ============================================================
# Evaluation helper functions
# ============================================================

def evaluate_classifier(model, loader, device):
    """Run inference and collect predictions + probabilities."""
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]  # prob of defective
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return np.array(all_preds), np.array(all_probs), np.array(all_labels)


def plot_confusion_matrix(y_true, y_pred, model_name, ax=None):
    cm = confusion_matrix(y_true, y_pred)
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Defective'],
                yticklabels=['Normal', 'Defective'])
    ax.set_title(f'Confusion Matrix — {model_name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    return cm


def plot_roc_curves(models_data, title='ROC Curve Comparison'):
    """
    models_data: list of (name, y_true, y_probs)
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
    
    for (name, y_true, y_probs), color in zip(models_data, colors):
        fpr, tpr, _ = roc_curve(y_true, y_probs)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../assets/roc_curves.png', dpi=150)
    plt.show()


def plot_precision_recall_curves(models_data):
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ['#e74c3c', '#3498db', '#2ecc71']
    
    for (name, y_true, y_probs), color in zip(models_data, colors):
        prec, rec, _ = precision_recall_curve(y_true, y_probs)
        ap = average_precision_score(y_true, y_probs)
        ax.plot(rec, prec, color=color, linewidth=2, label=f'{name} (AP = {ap:.3f})')
    
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('Precision-Recall Curve Comparison', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../assets/pr_curves.png', dpi=150)
    plt.show()


print('Evaluation functions defined.')

In [ ]:
# ============================================================
# Model Comparison Table
# ============================================================
def create_comparison_table(results_dict):
    """
    results_dict: {model_name: (y_true, y_pred, y_probs)}
    """
    from sklearn.metrics import accuracy_score
    rows = []
    for name, (y_true, y_pred, y_probs) in results_dict.items():
        fpr, tpr, _ = roc_curve(y_true, y_probs)
        report = classification_report(y_true, y_pred, output_dict=True)
        rows.append({
            'Model': name,
            'Accuracy': f"{accuracy_score(y_true, y_pred)*100:.1f}%",
            'Precision': f"{report['1']['precision']*100:.1f}%",
            'Recall': f"{report['1']['recall']*100:.1f}%",
            'F1': f"{report['1']['f1-score']*100:.1f}%",
            'AUROC': f"{auc(fpr, tpr):.3f}"
        })
    
    df = pd.DataFrame(rows).set_index('Model')
    print('\n=== Model Comparison ===')
    print(df.to_string())
    return df


# Demo output (replace with real results after training)
print('\nExpected output format:')
demo = pd.DataFrame([
    {'Model': 'ResNet-50', 'Accuracy': '96.2%', 'Precision': '95.8%', 'Recall': '96.7%', 'F1': '96.2%', 'AUROC': '0.989'},
    {'Model': 'Custom CNN', 'Accuracy': '91.4%', 'Precision': '90.1%', 'Recall': '92.3%', 'F1': '91.2%', 'AUROC': '0.961'},
    {'Model': 'Autoencoder', 'Accuracy': '88.7%', 'Precision': '87.2%', 'Recall': '90.1%', 'F1': '88.6%', 'AUROC': '0.943'},
]).set_index('Model')
print(demo.to_string())

In [ ]:
# ============================================================
# Threshold Analysis — Precision-Recall Tradeoff
# ============================================================
def plot_threshold_analysis(y_true, y_probs, model_name):
    """
    Shows how precision and recall change with classification threshold.
    Important for manufacturing: we prefer high recall.
    """
    thresholds = np.linspace(0.1, 0.9, 80)
    precisions, recalls, f1s = [], [], []
    
    for t in thresholds:
        preds = (y_probs >= t).astype(int)
        from sklearn.metrics import precision_score, recall_score
        precisions.append(precision_score(y_true, preds, zero_division=0))
        recalls.append(recall_score(y_true, preds, zero_division=0))
        f1s.append(f1_score(y_true, preds, zero_division=0))
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(thresholds, precisions, 'b-', label='Precision', linewidth=2)
    ax.plot(thresholds, recalls, 'r-', label='Recall', linewidth=2)
    ax.plot(thresholds, f1s, 'g-', label='F1', linewidth=2)
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.7, label='Default threshold')
    
    # Mark optimal F1 threshold
    best_t = thresholds[np.argmax(f1s)]
    ax.axvline(best_t, color='purple', linestyle=':', linewidth=2,
               label=f'Best F1 threshold={best_t:.2f}')
    
    ax.set_title(f'Threshold Analysis — {model_name}', fontsize=14)
    ax.set_xlabel('Classification Threshold')
    ax.set_ylabel('Score')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'../assets/threshold_{model_name}.png', dpi=150)
    plt.show()
    
    print(f'Best F1 at threshold {best_t:.2f}: F1={max(f1s):.3f}')

print('Threshold analysis function defined.')